In [ ]:
from pathlib import Path
import argparse

import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    pipeline,
)

In [ ]:
MODEL = "pranav-s/PolymerNER"

def load_ner():
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL,
        model_max_length=512,
    )

    model = AutoModelForTokenClassification.from_pretrained(MODEL)

    device = 0 if torch.cuda.is_available() else -1

    return pipeline(
        "ner",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="simple",
        device=device,
    )

In [5]:
def chunk_text(text, tokenizer, max_tokens=450):
    """
    Simple token-aware chunking.
    450 leaves some room below BERT's 512-token limit.
    """
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    for start in range(0, len(token_ids), max_tokens):
        chunk_ids = token_ids[start:start + max_tokens]

        yield tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
        )


def extract_property_names(markdown, ner):
    results = []

    for chunk in chunk_text(markdown, ner.tokenizer):
        entities = ner(chunk)

        for entity in entities:
            if entity["entity_group"] == "PROP_NAME":
                results.append(
                    {
                        "property": entity["word"],
                        "score": float(entity["score"]),
                    }
                )

    return results

In [7]:
paper_path = "../data/fulltext/arxiv/ID001097088/ID001097088.md"

markdown = Path(paper_path).read_text(
    encoding="utf-8"
)

ner = load_ner()
properties = extract_property_names(markdown, ner)

for x in properties:
    print(f'{x["property"]}\t{x["score"]:.3f}')

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (16446 > 512). Running this sequence through the model will result in indexing errors


diameter	0.529
polydispersity	0.772
m _ w	0.788
m _ w / m _ n	0.770
glass transition temperature	0.862
index	0.544
refractive index	0.929
n	0.897
e	0.502
h }	0.647
e }	0.654
h _ { \ text	0.623
lit } }	0.642
e _	0.677
\ text	0.588
lit } }	0.644


In [8]:
for chunk in chunk_text(markdown, ner.tokenizer):
    entities = ner(chunk)
    break

In [15]:
gen = chunk_text(markdown, ner.tokenizer)

In [24]:
chunk = next(gen)
print(chunk)
entities = ner(chunk)
entities

- w _ { \ text { ela } } \ quad ( 10 ) $ $ # # < span id = " page - 2 - 0 " > < / span > iii. experimental # # # a. sample preparation colloidal aggregates were prepared as thin films. compared to isolated aggregates such extended films are advantageous as the mechanical properties are not affected by finite size - effects and results from different indentation locations are well comparable. the base particles were made from poly ( methyl methacrylate ) ( pmma ) and synthesized via dispersion polymerization with heptane as solvent [ 42 ]. a comb - like graft copolymer with a backbone of methyl methacrylate and glycidyl methacrylate and teeth made from poly ( 12 - hydroxystearic acid ) [ 43 ] was copolymerized with the pmma and preferentially allocated at the surface of the forming particle. the poly ( 12 - hydroxystearic acid ) teeth served as a steric stabilization and prevented aggregation of the particles. moreover, nile red was added to the reaction mixture and homogeneously incorp

[{'entity_group': 'POLYMER',
  'score': np.float32(0.9683115),
  'word': 'poly ( methyl methacrylate )',
  'start': 429,
  'end': 457},
 {'entity_group': 'POLYMER',
  'score': np.float32(0.95561266),
  'word': 'pmma',
  'start': 460,
  'end': 464},
 {'entity_group': 'MONOMER',
  'score': np.float32(0.63050276),
  'word': 'methyl methacrylate',
  'start': 594,
  'end': 613},
 {'entity_group': 'MONOMER',
  'score': np.float32(0.65732247),
  'word': 'glycidyl methacrylate',
  'start': 618,
  'end': 639},
 {'entity_group': 'POLYMER',
  'score': np.float32(0.9326396),
  'word': 'poly ( 12 - hydroxystearic acid )',
  'start': 660,
  'end': 693},
 {'entity_group': 'POLYMER',
  'score': np.float32(0.9445986),
  'word': 'pmma',
  'start': 728,
  'end': 732},
 {'entity_group': 'POLYMER',
  'score': np.float32(0.9284822),
  'word': 'poly ( 12 - hydroxystearic acid )',
  'start': 806,
  'end': 839},
 {'entity_group': 'PROP_NAME',
  'score': np.float32(0.52863365),
  'word': 'diameter',
  'start': 